# 📊 TMS2 - Data Generation & Processing

Process uploaded Kaggle datasets to generate LSTM training data.

### Your Datasets:
| Dataset | Status | Files |
|---------|--------|-------|
| ⭐ UA-DETRAC | ✅ Partial | ~20K images |
| Real-Time Traffic | ✅ Complete | 500 videos |
| Singapore Density | ✅ Complete | 777 files |
| Serbia Traffic | ✅ Complete | - |
| LISA Traffic Light | ✅ Complete | - |

---

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/tms2_colab_training'
DATA_PATH = f'{DRIVE_PATH}/data'
KAGGLE_PATH = f'{DATA_PATH}/kaggle'

import os
os.makedirs(f'{DATA_PATH}/processed', exist_ok=True)
os.makedirs(f'{DATA_PATH}/lstm_ready', exist_ok=True)

print(f"Data path: {KAGGLE_PATH}")

Mounted at /content/drive
Data path: /content/drive/MyDrive/tms2_colab_training/data/kaggle


In [2]:
!pip install -q ultralytics opencv-python pandas numpy scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.4 MB/s eta 0:00:00


In [3]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import glob

print("Libraries loaded!")

Libraries loaded!


## 2. ⚙️ CONFIGURATION

**Edit these settings to control processing:**

In [4]:
# ============================================================
# ⚙️ CONFIGURATION - EDIT THESE VALUES
# ============================================================

# === PROCESSING LIMITS ===
# Set to smaller numbers for quick testing, None for ALL
MAX_IMAGES_PER_DATASET = None     # Max images to process per dataset
MAX_VIDEOS_PER_DATASET = None      # Max videos to process per dataset
MAX_FRAMES_PER_VIDEO = None        # Max frames per video
SAMPLE_INTERVAL = 10              # Process every Nth frame

# === WHICH DATASETS TO PROCESS ===
# Make sure it looks like this:
PROCESS_DATASETS = {
    'UA_DETRAC': True,
    'RealTime_Traffic_Videos': True,
    'Traffic_Density_Singapore': True,
    'RoadTraffic_Serbia': True,
    'LISA_Traffic_Light': True,
    'highway-traffic-videos': True,
}

# === DISPLAY CONFIG ===
print("📊 Processing Configuration:")
print(f"  Max images/dataset: {MAX_IMAGES_PER_DATASET}")
print(f"  Max videos/dataset: {MAX_VIDEOS_PER_DATASET}")
print(f"  Max frames/video: {MAX_FRAMES_PER_VIDEO}")
print(f"  Sample interval: Every {SAMPLE_INTERVAL} frames")
print("\n  Datasets:")
for name, enabled in PROCESS_DATASETS.items():
    print(f"    {'✅' if enabled else '❌'} {name}")

📊 Processing Configuration:
  Max images/dataset: None
  Max videos/dataset: None
  Max frames/video: None
  Sample interval: Every 10 frames

  Datasets:
    ✅ UA_DETRAC
    ✅ RealTime_Traffic_Videos
    ✅ Traffic_Density_Singapore
    ✅ RoadTraffic_Serbia
    ✅ LISA_Traffic_Light
    ✅ highway-traffic-videos


In [12]:
import os
KAGGLE_PATH = '/content/drive/MyDrive/tms2_colab_training/data/kaggle'

print("📂 Your folder structure:")
for item in os.listdir(KAGGLE_PATH):
    full_path = f'{KAGGLE_PATH}/{item}'
    if os.path.isdir(full_path):
        # Count files inside
        import glob
        files = glob.glob(f'{full_path}/**/*', recursive=True)
        files = [f for f in files if os.path.isfile(f)]
        print(f"  📁 {item}/ ({len(files)} files)")

        # Show first level subfolders
        for sub in os.listdir(full_path)[:5]:
            print(f"      └─ {sub}")

📂 Your folder structure:
  📁 720p-road-and-traffic-video/ (1 files)
      └─ 4K Road traffic video for object detection and tracking - free download now.mp4
  📁 temp/ (0 files)
  📁 highway-traffic-videos/ (254 files)
      └─ cctv052x2004080609x01863.avi
      └─ cctv052x2004080607x01842.avi
      └─ cctv052x2004080609x01872.avi
      └─ cctv052x2004080516x01639.avi
      └─ cctv052x2004080611x01906.avi


KeyboardInterrupt: 

In [10]:
import os
path = '/content/drive/MyDrive/tms2_colab_training/data/kaggle/RealTime_Traffic_Videos'
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    for f in files[:10]:  # Show first 10 files
        print(f'{indent}  └─ {f}')
    if len(files) > 10:
        print(f'{indent}  ... and {len(files)-10} more')

📁 RealTime_Traffic_Videos/
  └─ merged_annotations.json
  📁 real_traffic/
    └─ input-001.MOV
    └─ output.mp4


In [11]:
import os
path = '/content/drive/MyDrive/tms2_colab_training/data/kaggle/UA_DETRAC'
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    for f in files[:5]:
        print(f'{indent}  └─ {f}')
    if len(files) > 5:
        print(f'{indent}  ... and {len(files)-5} more files')

📁 UA_DETRAC/
  📁 archive/
    📁 content/
      📁 UA-DETRAC/
        📁 DETRAC_Upload/
          📁 labels/
            📁 train/
              └─ MVI_20052_img00541.txt
              └─ MVI_20052_img00542.txt
              └─ MVI_20052_img00516.txt
              └─ MVI_20052_img00458.txt
              └─ MVI_20052_img00473.txt
              ... and 6775 more files
            📁 val/
              └─ MVI_39311_img00116.txt
              └─ MVI_39311_img00124.txt
              └─ MVI_39311_img00089.txt
              └─ MVI_39311_img00078.txt
              └─ MVI_39311_img00054.txt
              ... and 6723 more files
          📁 images/
            📁 val/
              └─ MVI_39271_img01555.jpg
              └─ MVI_39311_img00008.jpg
              └─ MVI_39271_img01521.jpg
              └─ MVI_39271_img01483.jpg
              └─ MVI_39271_img01559.jpg
              ... and 6717 more files
            📁 train/
              └─ MVI_20052_img00426.jpg
              └─ MVI_20052_img00394.jpg
 

## 3. Discover Uploaded Datasets

In [5]:
# Your ACTUAL folder names (matched to what you uploaded)
DATASET_FOLDERS = {
    'UA_DETRAC': f'{KAGGLE_PATH}/UA_DETRAC',
    'RealTime_Traffic_Videos': f'{KAGGLE_PATH}/RealTime_Traffic_Videos',
    'Traffic_Density_Singapore': f'{KAGGLE_PATH}/Traffic_Density_Singapore',
    'RoadTraffic_Serbia': f'{KAGGLE_PATH}/RoadTraffic_Serbia',
    'LISA_Traffic_Light': f'{KAGGLE_PATH}/LISA_Traffic_Light',
    'highway-traffic-videos': f'{KAGGLE_PATH}/highway-traffic-videos',
}

discovered_data = {}

print("🔍 Discovering datasets (scanning ALL nested folders)...\n")

for name, path in DATASET_FOLDERS.items():
    if os.path.exists(path):
        # Recursively find ALL images and videos in nested folders
        videos = glob.glob(f'{path}/**/*.avi', recursive=True)
        videos += glob.glob(f'{path}/**/*.mp4', recursive=True)
        videos += glob.glob(f'{path}/**/*.mkv', recursive=True)
        videos += glob.glob(f'{path}/**/*.mov', recursive=True)
        videos += glob.glob(f'{path}/**/*.MOV', recursive=True)

        images = glob.glob(f'{path}/**/*.jpg', recursive=True)
        images += glob.glob(f'{path}/**/*.jpeg', recursive=True)
        images += glob.glob(f'{path}/**/*.png', recursive=True)

        if videos or images:
            discovered_data[name] = {
                'path': path,
                'videos': sorted(videos),
                'images': sorted(images)
            }
            print(f"✅ {name}:")
            print(f"   📁 {path}")
            print(f"   🖼️ {len(images)} images, 🎬 {len(videos)} videos")
        else:
            print(f"⚠️ {name}: Folder exists but no images/videos found")
    else:
        print(f"❌ {name}: Folder not found")

print(f"\n📊 Found {len(discovered_data)} datasets with data")

🔍 Discovering datasets (scanning ALL nested folders)...

✅ UA_DETRAC:
   📁 /content/drive/MyDrive/tms2_colab_training/data/kaggle/UA_DETRAC
   🖼️ 13452 images, 🎬 0 videos
✅ RealTime_Traffic_Videos:
   📁 /content/drive/MyDrive/tms2_colab_training/data/kaggle/RealTime_Traffic_Videos
   🖼️ 0 images, 🎬 2 videos
✅ Traffic_Density_Singapore:
   📁 /content/drive/MyDrive/tms2_colab_training/data/kaggle/Traffic_Density_Singapore
   🖼️ 4038 images, 🎬 0 videos
✅ RoadTraffic_Serbia:
   📁 /content/drive/MyDrive/tms2_colab_training/data/kaggle/RoadTraffic_Serbia
   🖼️ 71 images, 🎬 5 videos
✅ LISA_Traffic_Light:
   📁 /content/drive/MyDrive/tms2_colab_training/data/kaggle/LISA_Traffic_Light
   🖼️ 44075 images, 🎬 0 videos
✅ highway-traffic-videos:
   📁 /content/drive/MyDrive/tms2_colab_training/data/kaggle/highway-traffic-videos
   🖼️ 0 images, 🎬 254 videos

📊 Found 6 datasets with data


In [6]:
# Load YOLOv8
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print("YOLOv8 loaded!")

VEHICLE_CLASSES = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8 loaded!


In [7]:
def process_image(image_path, model):
    """Process single image with YOLOv8."""
    frame = cv2.imread(image_path)
    if frame is None:
        return None

    results = model(frame, verbose=False)

    counts = {'car': 0, 'truck': 0, 'bus': 0, 'motorcycle': 0}
    for result in results:
        for box in result.boxes:
            cls = int(box.cls[0])
            if cls in VEHICLE_CLASSES:
                counts[VEHICLE_CLASSES[cls]] += 1

    total = sum(counts.values())
    return {
        'vehicle_count': total,
        'cars': counts['car'],
        'trucks': counts['truck'],
        'buses': counts['bus'],
        'motorcycles': counts['motorcycle'],
        'traffic_density': min(1.0, total / 50),
        'estimated_speed': max(5, 40 - total * 0.8)
    }


def process_video(video_path, model, sample_interval=10, max_frames=None):
    """Process video with YOLOv8."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    records = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if max_frames and frame_count >= max_frames:
            break

        if frame_count % sample_interval == 0:
            results = model(frame, verbose=False)
            counts = {'car': 0, 'truck': 0, 'bus': 0, 'motorcycle': 0}

            for result in results:
                for box in result.boxes:
                    cls = int(box.cls[0])
                    if cls in VEHICLE_CLASSES:
                        counts[VEHICLE_CLASSES[cls]] += 1

            total = sum(counts.values())
            records.append({
                'frame': frame_count,
                'timestamp_sec': frame_count / fps,
                'vehicle_count': total,
                'cars': counts['car'],
                'trucks': counts['truck'],
                'buses': counts['bus'],
                'motorcycles': counts['motorcycle'],
                'traffic_density': min(1.0, total / 50),
                'estimated_speed': max(5, 40 - total * 0.8)
            })

        frame_count += 1

    cap.release()
    return records

In [13]:
import shutil
import os
import glob

LOCAL_DATA = '/content/local_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

# All your datasets
DATASETS_TO_COPY = [
    'UA_DETRAC',
    'Traffic_Density_Singapore',
    'RoadTraffic_Serbia',
    'LISA_Traffic_Light',
    'highway-traffic-videos',
    'RealTime_Traffic_Videos'
]

DRIVE_KAGGLE = '/content/drive/MyDrive/tms2_colab_training/data/kaggle'

print("📋 Copying datasets to local storage (one-time, faster processing)...\n")

for name in DATASETS_TO_COPY:
    src = f'{DRIVE_KAGGLE}/{name}'
    dst = f'{LOCAL_DATA}/{name}'

    if os.path.exists(src) and not os.path.exists(dst):
        print(f"📁 Copying {name}...")
        shutil.copytree(src, dst)
        print(f"   ✅ Done!")
    elif os.path.exists(dst):
        print(f"✅ {name} already local")
    else:
        print(f"❌ {name} not found in Drive")

print("\n✅ All datasets copied locally!")

# Update discovered_data to use local paths
for name in DATASETS_TO_COPY:
    local_path = f'{LOCAL_DATA}/{name}'
    if os.path.exists(local_path) and name in discovered_data:
        discovered_data[name]['path'] = local_path
        discovered_data[name]['images'] = glob.glob(f'{local_path}/**/*.jpg', recursive=True) + \
                                          glob.glob(f'{local_path}/**/*.png', recursive=True)
        discovered_data[name]['videos'] = glob.glob(f'{local_path}/**/*.avi', recursive=True) + \
                                          glob.glob(f'{local_path}/**/*.mp4', recursive=True) + \
                                          glob.glob(f'{local_path}/**/*.MOV', recursive=True)

# Verify
print("\n📊 Local data ready:")
for name, data in discovered_data.items():
    print(f"  {name}: {len(data['images'])} imgs, {len(data['videos'])} vids")

📋 Copying datasets to local storage (one-time, faster processing)...

📁 Copying UA_DETRAC...
   ✅ Done!
📁 Copying Traffic_Density_Singapore...
   ✅ Done!
📁 Copying RoadTraffic_Serbia...
   ✅ Done!
📁 Copying LISA_Traffic_Light...
   ✅ Done!
📁 Copying highway-traffic-videos...
   ✅ Done!
📁 Copying RealTime_Traffic_Videos...
   ✅ Done!

✅ All datasets copied locally!

📊 Local data ready:
  UA_DETRAC: 13452 imgs, 0 vids
  RealTime_Traffic_Videos: 0 imgs, 2 vids
  Traffic_Density_Singapore: 4027 imgs, 0 vids
  RoadTraffic_Serbia: 71 imgs, 0 vids
  LISA_Traffic_Light: 44075 imgs, 0 vids
  highway-traffic-videos: 0 imgs, 254 vids


## 4. Process All Datasets

⏱️ **Time estimate:** ~1-2 hours for your data size

In [14]:
from ultralytics import YOLO
import torch
import cv2
import os
from tqdm import tqdm

# ============================================================
# 🚀 GPU + BATCH OPTIMIZED PROCESSING
# ============================================================

# Force GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8n.pt')
model.to(device)
print(f"🎮 Using device: {device}")

VEHICLE_CLASSES = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
BATCH_SIZE = 256  # Reduce to 16 if OOM

def process_images_batch(image_paths, batch_size=32):
    """Process images in batches - 50x faster!"""
    records = []

    for i in tqdm(range(0, len(image_paths), batch_size), desc="Image batches"):
        batch_paths = image_paths[i:i+batch_size]
        results = model(batch_paths, verbose=False)

        for j, result in enumerate(results):
            counts = {'car': 0, 'truck': 0, 'bus': 0, 'motorcycle': 0}
            for box in result.boxes:
                cls = int(box.cls[0])
                if cls in VEHICLE_CLASSES:
                    counts[VEHICLE_CLASSES[cls]] += 1

            total = sum(counts.values())
            records.append({
                'source': os.path.basename(batch_paths[j]),
                'frame': i + j,
                'timestamp_sec': (i + j) * 5,
                'vehicle_count': total,
                'cars': counts['car'],
                'trucks': counts['truck'],
                'buses': counts['bus'],
                'motorcycles': counts['motorcycle'],
                'traffic_density': min(1.0, total / 50),
                'estimated_speed': max(5, 40 - total * 0.8)
            })
    return records


def process_video_fast(video_path, sample_interval=10, max_frames=500):
    """Process video with GPU - faster!"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    records = []
    frame_count = 0

    while frame_count < (max_frames or float('inf')):
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % sample_interval == 0:
            results = model(frame, verbose=False)

            counts = {'car': 0, 'truck': 0, 'bus': 0, 'motorcycle': 0}
            for result in results:
                for box in result.boxes:
                    cls = int(box.cls[0])
                    if cls in VEHICLE_CLASSES:
                        counts[VEHICLE_CLASSES[cls]] += 1

            total = sum(counts.values())
            records.append({
                'frame': frame_count,
                'timestamp_sec': frame_count / fps,
                'vehicle_count': total,
                'cars': counts['car'],
                'trucks': counts['truck'],
                'buses': counts['bus'],
                'motorcycles': counts['motorcycle'],
                'traffic_density': min(1.0, total / 50),
                'estimated_speed': max(5, 40 - total * 0.8)
            })

        frame_count += 1

    cap.release()
    return records


# ============================================================
# MAIN PROCESSING LOOP
# ============================================================
all_records = []

for name, data in discovered_data.items():
    if not PROCESS_DATASETS.get(name, False):
        print(f"⏭️ Skipping {name}")
        continue

    print(f"\n{'='*50}")
    print(f"🚀 Processing: {name}")
    print(f"{'='*50}")

    dataset_records = []

    # Process images (BATCHED)
    if data['images']:
        images = data['images'][:MAX_IMAGES_PER_DATASET] if MAX_IMAGES_PER_DATASET else data['images']
        print(f"🖼️ Processing {len(images)} images (batched)...")

        records = process_images_batch(images, BATCH_SIZE)
        for r in records:
            r['dataset'] = name
        dataset_records.extend(records)

    # Process videos
    if data['videos']:
        videos = data['videos'][:MAX_VIDEOS_PER_DATASET] if MAX_VIDEOS_PER_DATASET else data['videos']
        print(f"🎬 Processing {len(videos)} videos...")

        for video_path in tqdm(videos, desc="Videos"):
            records = process_video_fast(video_path, SAMPLE_INTERVAL, MAX_FRAMES_PER_VIDEO)
            for r in records:
                r['source'] = os.path.basename(video_path)
                r['dataset'] = name
            dataset_records.extend(records)

    all_records.extend(dataset_records)
    print(f"✅ {name}: {len(dataset_records)} records")

print(f"\n{'='*50}")
print(f"📊 TOTAL: {len(all_records)} records")
print(f"{'='*50}")

🎮 Using device: cuda

🚀 Processing: UA_DETRAC
🖼️ Processing 13452 images (batched)...


Image batches: 100%|██████████| 53/53 [03:19<00:00,  3.75s/it]


✅ UA_DETRAC: 13452 records

🚀 Processing: RealTime_Traffic_Videos
🎬 Processing 2 videos...


Videos: 100%|██████████| 2/2 [36:12<00:00, 1086.14s/it]


✅ RealTime_Traffic_Videos: 3127 records

🚀 Processing: Traffic_Density_Singapore
🖼️ Processing 4027 images (batched)...


...

✅ RoadTraffic_Serbia: 71 records

🚀 Processing: LISA_Traffic_Light
🖼️ Processing 44075 images (batched)...


Image batches: 100%|██████████| 173/173 [18:42<00:00,  6.49s/it]


✅ LISA_Traffic_Light: 44075 records

🚀 Processing: highway-traffic-videos
🎬 Processing 254 videos...


Videos: 100%|██████████| 254/254 [00:18<00:00, 13.40it/s]

✅ highway-traffic-videos: 1506 records

📊 TOTAL: 66258 records


In [15]:
# Save data
if all_records:
    df = pd.DataFrame(all_records)
    df.to_csv(f'{DATA_PATH}/processed/all_traffic_data.csv', index=False)

    print(f"✅ Saved {len(df)} records")
    print(f"\n📊 By dataset:")
    print(df.groupby('dataset')['vehicle_count'].agg(['count', 'mean']).round(2))
    display(df.head())

✅ Saved 66258 records

📊 By dataset:
                           count   mean
dataset                                
LISA_Traffic_Light         44075   5.01
RealTime_Traffic_Videos     3127   7.72
RoadTraffic_Serbia            71   6.77
Traffic_Density_Singapore   4027   9.34
UA_DETRAC                  13452  13.89
highway-traffic-videos      1506   6.15


,source,frame,timestamp_sec,vehicle_count,cars,trucks,buses,motorcycles,traffic_density,estimated_speed,dataset
0,MVI_20012_img00683.jpg,0,0.0,14,13,0,1,0,0.28,28.8,UA_DETRAC
1,MVI_20012_img00239.jpg,1,5.0,21,18,1,2,0,0.42,23.2,UA_DETRAC
2,MVI_20061_img00384.jpg,2,10.0,19,17,0,1,1,0.38,24.8,UA_DETRAC
3,MVI_20034_img00082.jpg,3,15.0,19,18,0,1,0,0.38,24.8,UA_DETRAC
4,MVI_20034_img00013.jpg,4,20.0,20,19,0,1,0,0.40,24.0,UA_DETRAC


## 5. Create LSTM Sequences

In [16]:
from sklearn.preprocessing import MinMaxScaler

# LSTM Configuration
SEQUENCE_LENGTH = 15
FEATURES = ['vehicle_count', 'traffic_density', 'estimated_speed', 'cars', 'trucks']

df = pd.read_csv(f'{DATA_PATH}/processed/all_traffic_data.csv')
available = [f for f in FEATURES if f in df.columns]
print(f"Using features: {available}")

# Scale
scaler = MinMaxScaler()
data = scaler.fit_transform(df[available])

target_scaler = MinMaxScaler()
targets = target_scaler.fit_transform(df['vehicle_count'].values.reshape(-1, 1)).flatten()

# Create sequences
X, y = [], []
for i in range(len(data) - SEQUENCE_LENGTH):
    X.append(data[i:i + SEQUENCE_LENGTH])
    y.append(targets[i + SEQUENCE_LENGTH])

X, y = np.array(X), np.array(y)
print(f"\n📊 LSTM Data: X={X.shape}, y={y.shape}")

Using features: ['vehicle_count', 'traffic_density', 'estimated_speed', 'cars', 'trucks']

📊 LSTM Data: X=(66243, 15, 5), y=(66243,)


In [17]:
# Save
np.save(f'{DATA_PATH}/lstm_ready/X_sequences.npy', X)
np.save(f'{DATA_PATH}/lstm_ready/y_targets.npy', y)

import pickle
with open(f'{DATA_PATH}/lstm_ready/feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(f'{DATA_PATH}/lstm_ready/target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)

print("✅ LSTM data saved!")

✅ LSTM data saved!


## ✅ Done!

**Next:** Run `02_LSTM_Training.ipynb`